In [ ]:
# ===== Cell 1 of 5 - set up: widgets, packages, the engine =====
# Run this first, and run it again after anything restarts Python. It is safe to run any number of times.
import importlib, importlib.metadata, importlib.util, os, re, subprocess, sys, tempfile

def notebook_folder():
    try:
        path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        return os.path.dirname(path if path.startswith("/Workspace") else "/Workspace" + path)
    except Exception:
        return os.getcwd()

HOME = notebook_folder()
ALLOW_DEFAULT_INDEX = False        # True only on a cluster meant to install from pip's own default index

# --- widgets: made before anything is installed, so this cell can run on a bare cluster
w = dbutils.widgets
def widget(name, default, label):
    try:
        return w.get(name)
    except Exception:
        w.text(name, default, label)
        return default
widget("llm_endpoint", "", "01 LLM endpoint"); widget("llm_token", "", "02 LLM token")
widget("reviewer_id", "", "03 Your user id (reviewer id, and the id sent to the LLM)")
widget("model_id", "", "04 Model ID"); widget("project", "", "05 Project date (empty = new project today)"); widget("run", "", "06 Run (empty = new run)")
widget("projects_dir", os.path.join(HOME, "Projects"), "07 Projects folder"); widget("jfrog_index_url", "", "08 Package index URL")
widget("concurrency_limit", "4", "09 Concurrency limit"); widget("token_cap", "40000", "10 Token cap")
widget("scratch_dir", "", "11 Scratch folder (usually empty)")
widget("concept_subject", "", "12 Subject of the documents, for concepts (e.g. financial)")
for old_widget in ("llm_user_id", "reviewer_role"):      # widgets of an earlier notebook, no longer used
    try:
        w.remove(old_widget)
    except Exception:
        pass

# --- packages: installed only if one is missing, pinned to what the runtime already has, and Python
# is restarted only if the runtime's own packages still import together afterwards
REQUIRED = ("yaml", "openpyxl", "docx", "numpy", "scipy", "sympy", "rdata")
missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]
if missing:
    index_url = w.get("jfrog_index_url").strip()
    requirements = os.path.join(HOME, "engine", "requirements.txt")
    if not index_url and not ALLOW_DEFAULT_INDEX:
        print("Packages missing:", ", ".join(missing), "- paste the package index URL into widget 08 and run this cell again.")
        print("Nothing is installed from an index you did not name. To use pip's default index, set ALLOW_DEFAULT_INDEX = True above.")
    else:
        pins = []
        for name in ("numpy", "pandas", "pyarrow", "scipy"):
            try:
                pins.append("%s==%s" % (name, importlib.metadata.version(name)))
            except importlib.metadata.PackageNotFoundError:
                pass
        constraints = os.path.join(tempfile.mkdtemp(prefix="verifier_"), "constraints.txt")
        open(constraints, "w").write("\n".join(pins) + "\n")
        hide = lambda text: re.sub(r"//[^/@\s]+@", "//...@", text or "")
        command = [sys.executable, "-m", "pip", "install", "-r", requirements, "-c", constraints] + (["--index-url", index_url] if index_url else [])
        print("Installing", ", ".join(missing), "| kept as the runtime has them:", ", ".join(pins) or "none found")
        done = subprocess.run(command, capture_output=True, text=True)
        if done.returncode:
            print("The install did not finish. What pip said:\n" + hide(done.stderr)[-1500:])
            print("If pip found no versions that fit, the index lacks an older version that works with this runtime's own packages. Nothing the runtime depends on was changed.")
        else:
            wanted = open(requirements, encoding="utf-8").read().split("# --- optional ---")
            if len(wanted) > 1:
                spare = os.path.join(tempfile.gettempdir(), "optional.txt")
                open(spare, "w", encoding="utf-8").write(wanted[1])
                extra = subprocess.run(command[:4] + ["-r", spare] + command[6:], capture_output=True, text=True)
                print("Optional packages (words inside pictures):", "installed." if not extra.returncode else "not installed; pictures of text will say so.")
            probe = subprocess.run([sys.executable, "-c", "import numpy, pandas, pyarrow"], capture_output=True, text=True)
            if probe.returncode:
                print("STOPPED BEFORE RESTARTING PYTHON: the runtime's own packages no longer import together (%s)." % hide((probe.stderr or "").strip().splitlines()[-1]))
                print("Restarting now would crash this notebook session. Detach this notebook from the cluster and attach it again to undo the install. Do not restart the cluster.")
            else:
                print("Installed. Restarting Python; then run this cell once more.")
                dbutils.library.restartPython()
else:
    # --- the engine
    for folder in (os.path.join(HOME, "engine"), os.path.join(HOME, "engine", "tests")):
        if folder not in sys.path:
            sys.path.insert(0, folder)
    import verifier
    if "LIVE" not in globals():
        LIVE = verifier.LiveValues()
    LIVE.update(w.get("llm_endpoint"), w.get("llm_token"), w.get("reviewer_id"))
    def live(name):
        """Read an endpoint, token or user id at the moment chat() is CALLED, so a fresh token pasted mid-run is used."""
        return LIVE.get(name)

    def current_settings():
        return verifier.make_settings({"concurrency_limit": int(w.get("concurrency_limit") or 4), "token_cap": int(w.get("token_cap") or 40000),
                                     "reviewer_id": w.get("reviewer_id"), "concept_subject": w.get("concept_subject").strip()})

    def open_current():
        """The run the widgets name, opened; or None with a message when the project has no inputs yet. Used
        by cells 3, 4 and 5, so that a cell run before cell 3 - or after Python restarted - says what to do."""
        _, missing = verifier.setup_project(w.get("projects_dir"), w.get("model_id"), w.get("project"))
        if missing:
            print("\n".join(missing)); print("Put the files in, then run cell 3.")
            return None
        if globals().get("PATHS") is not None and PATHS.model_id == w.get("model_id") and (not w.get("project") or PATHS.project_date == w.get("project")) and (not w.get("run") or PATHS.run_id == w.get("run")):
            return PATHS                               # keep working on the run this session opened
        return verifier.open_run(w.get("projects_dir"), w.get("model_id"), w.get("project"), w.get("run"), scratch_root=w.get("scratch_dir"))
    print("Folder:", HOME, "| Python", sys.version.split()[0], "| engine", verifier.ENGINE_VERSION)
    for name in REQUIRED + ("pdfplumber", "pypdf"):
        try:
            print("  %-11s %s" % (name, getattr(importlib.import_module(name), "__version__", "installed")))
        except Exception:
            print("  %-11s not installed%s" % (name, "" if name in ("pdfplumber", "pypdf") else " - run this cell again"))
    token = LIVE.get("llm_token")
    print("Endpoint set:", bool(LIVE.get("llm_endpoint")), "| token:", ("%d characters, pasted %.1f minutes ago" % (len(token), LIVE.token_age_minutes())) if token else "none pasted yet")
    print("Next: cell 2.")


In [ ]:
# ===== Cell 2 of 5 - your chat() =====
# Paste your organisation's chat() below, or leave USE_STANDIN = True to try the notebook without a model.
# chat(system_prompt, main_prompt) must return {"answer": "<the model's reply>"}. Read the endpoint, token and
# user id with live("...") INSIDE the function, so that a fresh token pasted into the widget is used mid-run.
USE_STANDIN = False

import requests

def chat(SystemPrompt, MainPrompt, history=[]):
    payload = {
        "app": "sparkair",
        "enable_streaming": False,
        "flow_name": "general_chat",
        "history": history,
        "optionalParameter": {
            "maxtoken": 250000,
            "contextlength": 250000,
            "Temperature": 0.01,           # low: the same question gives the same answer
            "Top_k": 1,
            "Penalty": 1.1,
            "DefaultPrompt": SystemPrompt,
        },
        "query": MainPrompt,
        "select_all": False,
    }
    headers = {
        "Authorization": f'Bearer {live("llm_token")}',
        "SP_SSO_UID": live("reviewer_id"),
        "Content-Type": "application/json",
    }
    response = requests.post(live("llm_endpoint"), json=payload, headers=headers, timeout=180)
    response.raise_for_status()
    return response.json()          # the engine reads the reply from "answer", or from an OpenAI-shaped "choices"

if USE_STANDIN:
    import standin_chat
    ACTIVE_CHAT = standin_chat.chat
    print("Using the stand-in: no model is called; answers are made up from the prompt, for trying the notebook only.")
else:
    ACTIVE_CHAT = chat
    try:
        reply = ACTIVE_CHAT("Reply with the single word OK.", "Reply with the single word OK.")["answer"]
        print("chat() answered:", reply[:60], "| Next: cell 3.")
    except Exception as problem:
        print("chat() did not answer (%s: %s). Check widgets 01 to 03, or set USE_STANDIN = True to try without a model." % (type(problem).__name__, str(problem)[:120]))


In [ ]:
# ===== Cell 3 of 5 - read the inputs (no model involved) =====
# Makes the project folder if it is new, tells you what to put where, reads every input file, and shows
# the outline of the methodology for you to check before any model call is spent.
PATHS = open_current()
if PATHS is not None:
    SETTINGS = current_settings()
    RESULT = verifier.run_pipeline(PATHS, SETTINGS, chat=None, live=LIVE, stop_after="06")
    print(RESULT["message"])
    store = verifier.open_store(PATHS, SETTINGS)
    for outline in store.read("outline"):
        if outline["corner"] == "canon":
            print("\nOutline of the methodology as it was read (first 60 lines):\n" + "\n".join(outline["lines"][:60]))
    for record in store.read("step_records"):
        for message in record["messages"]:
            if message.startswith(("Read as:", "Not read:")) or "left out" in message or "R package" in message:
                print("  " + message)
    flow = store.read("dataflow")
    if flow:
        outputs, how, _ = verifier.decided_outputs(flow, {})
        print("\nFinal outputs code proposes: %s." % "; ".join("%s (%s)" % (name, how[name]) for name in outputs))
    concepts, _ = verifier.latest_concepts(store.read)
    print("\nConcepts found by code: %d (sheet Concepts). The model refines them after you confirm the outline." % len(concepts))
    print("Run folder:", PATHS.run_dir)
    print("Open Output.xlsx there. The three Chunks sheets show everything that was read; Model_Package_Info what was not.")
    print("into the run folder before cell 4. If the outline is right, go to cell 4; if not, fix the input and run this cell again.")


In [ ]:
# ===== Cell 4 of 5 - confirm the outline, then run the model steps and the checks =====
# First run: records that you confirmed the outline, then starts the model steps in the background.
# Run it again at any time to see where the run stands. Set PAUSE or STOP to True and run it to pause or stop.
OUTLINE_CONFIRMED = True      # set False if you have not checked the outline in cell 3
PAUSE, STOP = False, False
MODE = "A"                    # "A": background thread. "C": foreground, stops by itself after FOREGROUND_MINUTES.
FOREGROUND_MINUTES = 12
import threading

def keep_alive():
    """A trivial Spark action so the cluster does not shut down mid-run - in its own thread, so that
    a slow or stuck Spark call can never hold the review up."""
    def touch():
        try:
            spark.range(1).count()
        except Exception:
            pass
    threading.Thread(target=touch, daemon=True).start()

def work():
    """The run itself. Whatever goes wrong is written into RESULT, where cell 4 shows it: a thread
    that dies otherwise dies in silence, and the run just looks idle."""
    import traceback
    try:
        settings = dict(SETTINGS, foreground_minutes=FOREGROUND_MINUTES if MODE == "C" else 0.0, token_wait="stop" if MODE == "C" else "wait")
        RESULT.update(verifier.run_pipeline(PATHS, verifier.make_settings({k: v for k, v in settings.items() if v != verifier.DEFAULT_SETTINGS.get(k)}),
                                          chat=ACTIVE_CHAT, live=LIVE, state=STATE, keep_alive=keep_alive))
    except Exception as problem:
        RESULT.update(state="failed", message="The run stopped: %s: %s" % (type(problem).__name__, problem),
                      details=traceback.format_exc())

RESULT = globals().get("RESULT") or {}

if "PATHS" not in globals() or PATHS is None:
    PATHS = open_current()
    SETTINGS = current_settings() if PATHS is not None else None
if PATHS is None:
    print("Cell 3 has not read the inputs yet. Run cell 3 first.")
elif "STATE" not in globals():
    store = verifier.open_store(PATHS, SETTINGS)
    if not OUTLINE_CONFIRMED:
        print("Check the outline in cell 3 first, then set OUTLINE_CONFIRMED = True.")
    else:
        print(verifier.confirm_outline(PATHS, SETTINGS, w.get("reviewer_id")))
        STATE = verifier.AskState()
        if MODE == "C":
            work(); print(RESULT["message"])
        else:
            WORKER = threading.Thread(target=work, name="verifier-run", daemon=True); WORKER.start()
            print("The run works in the background. Run this cell again to see where it stands; paste a fresh token into widget 02 whenever it asks.")
else:
    store = verifier.open_store(PATHS, SETTINGS)
    STATE.control["pause"], STATE.control["stop"] = PAUSE, STOP
    print(verifier.progress_text(store, RESULT.get("message", "")))
    for record in store.read("step_records"):
        print("  step %s %-22s %s" % (record["step_id"], record["name"], ", ".join("%s: %s" % item for item in sorted(record["counts"].items()))))
    for label, value in verifier.call_statistics(store):
        print("  %-52s %s" % (label, value))
    print("Token pasted %.1f minutes ago." % LIVE.token_age_minutes())
    if STATE.waiting_for_token:
        print("WAITING FOR A FRESH TOKEN: the gateway refused the last call. Paste a new token into widget 02 and run cell 1; the run goes on by itself.")
    alive = "WORKER" in globals() and WORKER.is_alive()
    if RESULT.get("state") == "failed":
        print("THE RUN STOPPED WITH A PROBLEM:", RESULT["message"])
        print(RESULT.get("details", "")[-1500:])
        print("Fix what it says, then set STATE aside (del STATE) and run this cell again; finished steps are not repeated.")
    elif alive:
        print("The run is working in the background. Run this cell again to see progress.")
    else:
        print("The run is not working at the moment:", RESULT.get("message", "no message yet"), "| When it waits for a person, go to cell 5.")


In [ ]:
# ===== Cell 5 of 5 - finish: verify the evidence pack =====
# Run this when the run has finished, to check the run folder against its own record: the inputs are the
# files that were read, the engine is the one that produced it, re-reading gives the same content, the
# graph chain verifies, and no access token was written anywhere in the folder.
# APPENDIX runs a maintainer's check instead: "probe" (a new cluster), "sanity" (the sample projects with the
# stand-in), "harness" (seeded differences), "sign-off" (guided reading against your real chat()), or
# "map-sign-off" (the implementation map's agents against your real chat()).
APPENDIX = ""

if APPENDIX:
    import develop, standin_chat
    if APPENDIX == "probe":
        print(develop.run_probe(w.get("projects_dir"), chat=ACTIVE_CHAT, live=LIVE))
    elif APPENDIX == "sanity":
        import datetime, shutil, tempfile
        demo = tempfile.mkdtemp(prefix="verifier_sanity_")
        shutil.copytree(os.path.join(HOME, "engine", "tests", "sample_projects", "A_minimal", "Inputs"), os.path.join(demo, "SANITY", datetime.date.today().isoformat(), "Inputs"))
        demo_paths = verifier.open_run(demo, "SANITY")
        print(verifier.run_pipeline(demo_paths, verifier.make_settings({"require_outline_confirmation": False}), chat=standin_chat.chat)["message"])
        print("Open", os.path.join(demo_paths.run_dir, "Output.xlsx"))
    elif APPENDIX == "harness":
        develop.harness(["A_minimal", "--limit", "10"])
    elif APPENDIX == "sign-off":
        print(develop.run(ACTIVE_CHAT, LIVE, label="the real model"))
    elif APPENDIX == "map-sign-off":
        print(develop.map_report(ACTIVE_CHAT, LIVE, label="the real model"))
elif "PATHS" not in globals() or PATHS is None:
    print("Cell 3 has not read the inputs yet. Run cells 3 and 4 first.")
else:
    SETTINGS = current_settings()
    RESULT = verifier.run_pipeline(PATHS, SETTINGS, chat=ACTIVE_CHAT if "ACTIVE_CHAT" in globals() else None, live=LIVE)
    print(RESULT["message"])
    print("\nVerifying the evidence pack:")
    for what, verdict, detail in verifier.verify_evidence_pack(PATHS, SETTINGS, live=LIVE):
        print("  %-34s %-10s %s" % (what, verdict, detail))
    print("\nRun folder:", PATHS.run_dir, "- Output.xlsx and Validation_Report.docx are the deliverables; _audit/ holds the record.")
